In [ ]:
import pandas as pd
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# --- Load your data ---
pos = pd.read_csv("/rhome/zli529/lab/LncRNA_chip_prediction/Final/feature_selection/positives_TSS.csv")
neg = pd.read_csv("/rhome/zli529/lab/LncRNA_chip_prediction/Final/feature_selection/negatives_TSS.csv")

# Add labels
pos["label"] = 1
neg["label"] = 0

# Combine
data = pd.concat([pos, neg], ignore_index=True)

# --- Features / Labels ---
X = data.drop(columns=["label"])
X = X.select_dtypes(include=["number"])   # keep only numeric features
y = data["label"]

print("Original features:", X.shape[1])

# --- Step 1: Remove low-variance features ---
var_thresh = VarianceThreshold(threshold=0.01)   # tune threshold if needed
X_var = var_thresh.fit_transform(X)

print("After variance threshold:", X_var.shape[1])

# --- Step 2: Univariate feature selection (ANOVA F-test) ---
# Keep top 30 features (adjust 'k' depending on dataset size)
uni_selector = SelectKBest(score_func=f_classif, k=40)
X_uni = uni_selector.fit_transform(X_var, y)

print("After univariate selection:", X_uni.shape[1])

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X_uni, y, test_size=0.3, stratify=y, random_state=42
)

# --- Train a classifier (example: RandomForest) ---
clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

# --- Evaluate ---
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))
